A.) Dynamic Goal-Based Agent for Warehouse Logistics Optimization.
A robotic agent operates in a warehouse modeled as an N×M grid environment. The agent starts at a predefined loading dock and must deliver packages to multiple destinations marked on the grid while avoiding dynamically placed obstacles.
Take suitable values of the following parameters.
- Warehouse dimensions: N×M grid size (M,N between 5 and 10, inclusive)
- Number of packages: P (between 2 and 6, inclusive)
- Number of obstacles: O (between 1 and 10, inclusive)
- Package locations: (X1, Y1), (X2, Y2), ... (XP, YP)
- Drop-off locations: (D1X, D1Y), (D2X, D2Y), ... (DPX, DPY)
- Robot starting position: S=(Sx,Sy), starts at a fixed cell but moves dynamically
- Movement cost: Each movement incurs a cost of 1 unit
- Delivery reward: Successfully delivering a package adds 10 units to the total reward - Obstacle penalty: Hitting an obstacle results in a (-5) penalty

**Note :** Packages locations and drop-off locations should not overlap.

**Q1. Represent the warehouse as an N×M matrix. Place the packages, drop-off points, and obstacles randomly. Display the initial warehouse configuration.**

In [15]:
import numpy as np
import random
from collections import deque

def setup_warehouse( start_point_x, start_point_y, grid_rows, grid_cols, num_packages, num_obstacles,random_seed=42):
    """Initialize the warehouse environment with obstacles, packages, and drop-off points."""
    random.seed(random_seed)
    np.random.seed(random_seed)

    warehouse_grid = np.full((grid_rows, grid_cols), '.', dtype=str)

    # Randomly placing obstacles
    obstacle_positions = set()
    while len(obstacle_positions) < num_obstacles:
        obs_x, obs_y = random.randint(0, grid_rows - 1), random.randint(0, grid_cols - 1)
        if (obs_x, obs_y) not in obstacle_positions:
            obstacle_positions.add((obs_x, obs_y))
            warehouse_grid[obs_x, obs_y] = 'X'

    # Assign packages and delivery locations
    package_locations = {}
    delivery_spots = {}

    while len(package_locations) < num_packages:
        pkg_x, pkg_y = random.randint(0, grid_rows - 1), random.randint(0, grid_cols - 1)
        drop_x, drop_y = random.randint(0, grid_rows - 1), random.randint(0, grid_cols - 1)

        if (pkg_x, pkg_y) not in obstacle_positions and (drop_x, drop_y) not in obstacle_positions and (pkg_x, pkg_y) not in delivery_spots and (drop_x, drop_y) not in package_locations and (
                pkg_x, pkg_y) != (drop_x, drop_y):
            package_locations[(pkg_x, pkg_y)] = (drop_x, drop_y)
            delivery_spots[(drop_x, drop_y)] = (pkg_x, pkg_y)
            warehouse_grid[pkg_x, pkg_y] = 'P'
            warehouse_grid[drop_x, drop_y] = 'D'

    # Robot starting position
    start_x, start_y = start_point_x , start_point_y
    warehouse_grid[start_x, start_y] = 'R'

    return warehouse_grid, package_locations, delivery_spots, obstacle_positions, (start_x, start_y)

def display_warehouse(grid):
    """Prints the warehouse layout."""
    for row in grid:
        print(" ".join(row))
    print()

def execute_robot(start_point_x, start_point_y, grid_rows, grid_cols, num_packages, num_obstacles,random_seed=42):
    """Controls the agent's movement, calculates path costs, and evaluates performance."""
    warehouse, packages, drop_locations, obstacle_set, robot_position = setup_warehouse(start_point_x, start_point_y, grid_rows, grid_cols,
                                                                                        num_packages, num_obstacles,
                                                                                        random_seed)

    print("Initial Warehouse Setup:")
    display_warehouse(warehouse)

N = 6
M = 6
P = 3
D = 3
O = 5
start_point_x = 0
start_point_y = 0


# Running the warehouse simulation with specified parameters
execute_robot(start_point_x, start_point_y, grid_rows=N, grid_cols=M, num_packages=P, num_obstacles=O, random_seed=42)

Initial Warehouse Setup:
R D . . . X
. X . . . .
. X . . . .
. D . . . .
P . . D . .
X . . . P X



**Q2. Implement a goal-based agent that can identify all goals, plan a sequence of actions to reach the goal, use a search algorithm (BFS, DFS, or UCS) to find optimal paths, deliver all packages, and calculate the total cost.**

In [22]:
def bfs_shortest_path(start, destination, obstacles, grid_rows, grid_cols):
    """Implements Breadth-First Search (BFS) to determine the shortest path."""
    move_directions = [(-1, 0), (1, 0), (0, -1), (0, 1)]
    queue = deque([(start, [])])
    visited = set()

    while queue:
        (cur_x, cur_y), path_history = queue.popleft()

        if (cur_x, cur_y) in visited:
            continue
        visited.add((cur_x, cur_y))

        if (cur_x, cur_y) == destination:
            return path_history + [(cur_x, cur_y)], len(path_history)

        for move_x, move_y in move_directions:
            next_x, next_y = cur_x + move_x, cur_y + move_y

            if 0 <= next_x < grid_rows and 0 <= next_y < grid_cols :
                queue.append(((next_x, next_y), path_history + [(cur_x, cur_y)]))

    return None, float('inf')  # No possible path


def execute_robot(start_point_x, start_point_y, grid_rows, grid_cols, num_packages, num_obstacles, random_seed=42):
    """Controls the agent's movement, calculates path costs, and evaluates performance."""
    warehouse, packages, drop_locations, obstacle_set, robot_position = setup_warehouse(start_point_x, start_point_y, grid_rows, grid_cols,
                                                                                        num_packages, num_obstacles,
                                                                                        random_seed)

    print("Initial Warehouse Setup:")
    display_warehouse(warehouse)

    total_movement_cost = 0
    delivery_score = 0
    obstacle_penalty = 0
    robot_location = robot_position

    for package_site, drop_site in packages.items():
        path_to_package, cost_to_package = bfs_shortest_path(robot_location, package_site, obstacle_set, grid_rows,
                                                             grid_cols)
        path_to_delivery, cost_to_delivery = bfs_shortest_path(package_site, drop_site, obstacle_set, grid_rows,
                                                               grid_cols)

        if path_to_package and path_to_delivery:
            print(f"[START] Route to package at {package_site}: {path_to_package}")
            print(f"[END] Delivery to {drop_site}: {path_to_delivery}")
            total_movement_cost += (cost_to_package + cost_to_delivery)
            delivery_score += 10  # Reward for successful delivery
            robot_location = drop_site  # Update robot’s current position

            # Check if any obstacle is hit
            for step in path_to_package + path_to_delivery:
                if step in obstacle_set:
                    obstacle_penalty -= 5

    final_evaluation = delivery_score - total_movement_cost + obstacle_penalty
    print(
        f"Total Movement Cost: {total_movement_cost}, Total Delivery Reward: {delivery_score}, Obstacle Penalty: {obstacle_penalty}, Final Score: {final_evaluation}")

    #print("Final Warehouse:")
    #display_warehouse(warehouse)

N = 6
M = 6
P = 3
D = 3
O = 5
D = P
start_point_x = 0
start_point_y = 0


# Running the warehouse simulation with specified parameters
execute_robot(start_point_x, start_point_y, grid_rows=N, grid_cols=M, num_packages=P, num_obstacles=O, random_seed=42)

Initial Warehouse Setup:
R D . . . X
. X . . . .
. X . . . .
. D . . . .
P . . D . .
X . . . P X

[START] Route to package at (4, 0): [(0, 0), (1, 0), (2, 0), (3, 0), (4, 0)]
[END] Delivery to (4, 3): [(4, 0), (4, 1), (4, 2), (4, 3)]
[START] Route to package at (0, 0): [(4, 3), (3, 3), (2, 3), (1, 3), (0, 3), (0, 2), (0, 1), (0, 0)]
[END] Delivery to (0, 1): [(0, 0), (0, 1)]
[START] Route to package at (5, 4): [(0, 1), (1, 1), (2, 1), (3, 1), (4, 1), (5, 1), (5, 2), (5, 3), (5, 4)]
[END] Delivery to (3, 1): [(5, 4), (4, 4), (3, 4), (3, 3), (3, 2), (3, 1)]
Total Movement Cost: 28, Total Delivery Reward: 30, Obstacle Penalty: -10, Final Score: -8


**Q3. Choose a random seed value for the ease of reproducing the results. Your program should give outputs: the chosen path taken by the agent, total cost and rewards, final score based on penalties, movement costs, and successful deliveries.**

In [29]:
import random

N = random.randint(5, 10)  # Warehouse rows
M = random.randint(5, 10)  # Warehouse columns
P = random.randint(2, 6)   # Number of packages
O = random.randint(1, 10)  # Number of obstacles
D = P  # Number of drop-off locations (equal to P)
start_point_x = random.randint(0, N-1) # Robot start point row
start_point_y = random.randint(0, M-1) # Robot start point column

# Running the warehouse simulation with specified parameters
execute_robot(start_point_x, start_point_y, grid_rows=N, grid_cols=M, num_packages=P, num_obstacles=O, random_seed=42)

Initial Warehouse Setup:
D . D . . . . . .
X . . P . . P . X
. . . . . . . . .
. . X . . . . D D
. . . X . . . . .
. . . . . . . . .
. . . . . . . . .
. . . . R . . . .
. . . D . . P . .
P . . . P . . . .

[START] Route to package at (1, 6): [(7, 4), (6, 4), (5, 4), (4, 4), (3, 4), (2, 4), (1, 4), (1, 5), (1, 6)]
[END] Delivery to (0, 0): [(1, 6), (0, 6), (0, 5), (0, 4), (0, 3), (0, 2), (0, 1), (0, 0)]
[START] Route to package at (1, 3): [(0, 0), (1, 0), (1, 1), (1, 2), (1, 3)]
[END] Delivery to (3, 8): [(1, 3), (2, 3), (3, 3), (3, 4), (3, 5), (3, 6), (3, 7), (3, 8)]
[START] Route to package at (9, 0): [(3, 8), (4, 8), (5, 8), (6, 8), (7, 8), (8, 8), (9, 8), (9, 7), (9, 6), (9, 5), (9, 4), (9, 3), (9, 2), (9, 1), (9, 0)]
[END] Delivery to (8, 3): [(9, 0), (8, 0), (8, 1), (8, 2), (8, 3)]
[START] Route to package at (8, 6): [(8, 3), (8, 4), (8, 5), (8, 6)]
[END] Delivery to (3, 7): [(8, 6), (7, 6), (6, 6), (5, 6), (4, 6), (3, 6), (3, 7)]
[START] Route to package at (9, 4): [(3, 7), (4, 7